# 03 — Kiểm định bộ dữ liệu (`training/validate_dataset.py`)

**Nhiệm vụ ClickUp:** Kiểm định bộ dữ liệu bằng validate_dataset.py (tiện ích cho Tuần 2 và Việc 5)

**Mục tiêu:** một notebook tiện ích dùng lại nhiều lần: điền danh sách file `.jsonl` → chạy cổng kiểm định
`training/validate_dataset.py` (nguồn sự thật duy nhất cho hợp đồng 1) → xem số hàng, exit code, số lỗi theo loại,
3 hàng lỗi đầu tiên (nếu có) và lời giải thích từng loại lỗi. Dùng ở **Tuần 2** (hàng công khai xLAM) và sau này ở
**Việc 5** (hàng SGOD với `--tools tools/sgod/sgod_tools.json`).

Notebook **không** cài đặt lại logic kiểm định (quy tắc trong `docs/contracts/cli.md`); ô "định vị hàng lỗi" chỉ giúp tìm hàng để mở xem.

> Notebook này **chỉ dùng stdlib** (`json`, `re`, `subprocess`) nên **cố ý không có ô `pip install` và ô kiểm tra GPU/phiên bản**
> như các notebook khác — ô Thiết lập chung ở dưới là đủ; chạy được trên CPU, Colab hay local.

In [ ]:
# --- Thiết lập (Colab + local) --------------------------------------------------------------
# Colab : read GITHUB_TOKEN from Secrets, clone the private repo (skip if present), chdir into it.
# Local : walk up from the current directory until the repo root (docs/contracts/cli.md) is found.
# Sets REPO (Path), GIT_SHA, IN_COLAB, AUTHOR and a run() helper that calls repo scripts.
import os, shlex, subprocess, sys
from pathlib import Path

REPO_HTTPS = "github.com/thanhhao98/ChatSystem"
MARKER = "docs/contracts/cli.md"          # exists at the root of every checkout

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))


def _github_token():
    # Colab Secrets -> Kaggle Secrets -> environment variable. Never print the value.
    if IN_COLAB:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    if IN_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    return os.environ["GITHUB_TOKEN"]


if IN_COLAB or IN_KAGGLE:
    try:
        _token = _github_token()
    except Exception as e:  # SecretNotFoundError / NotebookAccessError / KeyError
        raise RuntimeError(
            "Thiếu secret GITHUB_TOKEN. Colab: biểu tượng chìa khoá (Secrets) -> Add new secret: "
            "Name = GITHUB_TOKEN, Value = Personal access token (classic, scope repo) của tài khoản collaborator "
            "trên thanhhao98/ChatSystem, bật 'Notebook access'. Kaggle: Add-ons -> Secrets -> GITHUB_TOKEN. "
            "Rồi chạy lại ô này.") from e
    if not Path("ChatSystem").exists():
        _r = subprocess.run(["git", "clone", "--quiet", f"https://{_token}@{REPO_HTTPS}", "ChatSystem"],
                            capture_output=True, text=True)
        if _r.returncode != 0:
            raise RuntimeError("git clone thất bại: " + _r.stderr.replace(_token, "<token>"))
    os.chdir("ChatSystem")
    del _token
else:
    _here = Path.cwd().resolve()
    for _cand in [_here, *_here.parents]:
        if (_cand / MARKER).exists():
            os.chdir(_cand)
            break
    else:
        raise FileNotFoundError(f"Không tìm thấy gốc repo (không có {MARKER}) khi đi lên từ {_here}. "
                                "Mở notebook từ bên trong thư mục ChatSystem đã clone.")

REPO = Path.cwd()


def _git(*args):
    # Small helper: run a git command in REPO and return stdout ("" on any failure).
    try:
        return subprocess.run(["git", *args], cwd=REPO, capture_output=True, text=True).stdout.strip()
    except OSError:
        return ""


GIT_SHA = _git("rev-parse", "--short", "HEAD") or "no-git"
AUTHOR = os.environ.get("GITHUB_USER") or _git("config", "user.name") or "điền tên"


def run(cmd):
    # Run a repo script (list of args), echo the command, stream its output, return CompletedProcess.
    shown = " ".join(shlex.quote(c) for c in cmd).replace(shlex.quote(sys.executable), "python", 1)
    print("$ " + shown)
    p = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
    if p.stdout:
        print(p.stdout.rstrip())
    if p.stderr:
        print(p.stderr.rstrip())
    print(f"[exit code = {p.returncode}]")
    return p


print(f"REPO      = {REPO}")
print(f"git HEAD  = {GIT_SHA}")
print(f"python    = {sys.version.split()[0]} · Colab = {IN_COLAB} · Kaggle = {IN_KAGGLE} · author = {AUTHOR}")

## Tham số

Sửa `FILES` (đường dẫn tương đối gốc repo hoặc tuyệt đối). Hàng công khai có `replay_tools` → để `TOOLS` rỗng.
Hàng SGOD không có `replay_tools` → điền `TOOLS = "tools/sgod/sgod_tools.json"` và `ROLES = "tools/sgod/roles.json"`,
đổi `PREAMBLE` sang `prompts/system_preamble_v1.txt`.

In [ ]:
# @title Tham số { display-mode: "form" }
# Files to validate — edit this list, then Runtime -> Run all.
FILES = [
    "data/public/xlam_2k.train.jsonl",
    "data/public/xlam_2k.val.jsonl",
    "data/public/xlam_2k.test.jsonl",
]
PREAMBLE = "prompts/system_preamble_v0.txt"  # @param {type:"string"}   empty string -> --no-preamble-check
TOOLS = ""  # @param {type:"string"}   e.g. tools/sgod/sgod_tools.json for rows WITHOUT replay_tools
ROLES = ""  # @param {type:"string"}   e.g. tools/sgod/roles.json (default vocabulary: employee, company_admin, system_admin)
FIXED_REPLIES = "prompts/fixed_replies.json"  # @param {type:"string"}
MAX_ROWS = 0  # @param {type:"integer"}   0 = all rows

## Chạy cổng kiểm định

In [ ]:
# Build the command from the form values (flag names are fixed by docs/contracts/cli.md) and run it.
import json
from pathlib import Path

cmd = [sys.executable, "training/validate_dataset.py", *FILES]
cmd += ["--preamble", PREAMBLE] if PREAMBLE else ["--no-preamble-check"]
if TOOLS:
    cmd += ["--tools", TOOLS]
if ROLES:
    cmd += ["--roles", ROLES]
if FIXED_REPLIES:
    cmd += ["--fixed-replies", FIXED_REPLIES]
if MAX_ROWS:
    cmd += ["--max-rows", str(MAX_ROWS)]

ROW_COUNTS = {f: sum(1 for l in (REPO / f).open(encoding="utf-8") if l.strip()) for f in FILES}
for f, n in ROW_COUNTS.items():
    print(f"{n:6d} rows  {f}")

VALIDATOR = REPO / "training/validate_dataset.py"
EXIT_CODE, VALIDATOR_OUTPUT = None, ""
if VALIDATOR.exists():
    p_val = run(cmd)
    EXIT_CODE, VALIDATOR_OUTPUT = p_val.returncode, (p_val.stdout + p_val.stderr)
else:
    print("Chưa có training/validate_dataset.py trong bản checkout này -> git pull (hoặc chờ PR tương ứng được merge) rồi chạy lại ô này.")
print("exit code =", EXIT_CODE, "(0 = hợp lệ)")

## Định vị 3 hàng lỗi đầu tiên (nếu có)

Kết quả của `validate_dataset.py` ở trên là **nguồn sự thật**. Ô này chỉ quét nhanh các quy tắc cấu trúc của hợp đồng 1
để chỉ ra *hàng nào* cần mở xem; nếu nó tìm thấy 0 hàng nhưng script vẫn báo lỗi, đọc thông báo của script (ví dụ lỗi tham số theo schema).

In [ ]:
# Light structural pre-check that points at offending rows (not a replacement for the validator).
# Error kinds use the same names as training/validate_dataset.py.
import re

TAG = re.compile(r"<tool_call>(.*?)</tool_call>", re.S)
preamble_text = (REPO / PREAMBLE).read_text(encoding="utf-8").strip() if PREAMBLE else None
fixed_replies = set(json.load(open(REPO / FIXED_REPLIES, encoding="utf-8")).values()) if FIXED_REPLIES else set()
roles_ok = {"employee", "company_admin", "system_admin"}
if ROLES:  # tools/sgod/roles.json = {"tiers": {...}, "roles": [...]}
    _r = json.load(open(REPO / ROLES, encoding="utf-8"))
    roles_ok = set(_r["roles"]) if isinstance(_r, dict) and isinstance(_r.get("roles"), list) else set(_r)
catalogue = None  # name -> set(parameter names), from --tools
if TOOLS:
    catalogue = {t["function"]["name"]: set(((t["function"].get("parameters") or {}).get("properties") or {}))
                 for t in json.load(open(REPO / TOOLS, encoding="utf-8"))}


def row_problems(row):
    probs = []
    if not isinstance(row, dict):
        return ["json_invalid"]
    if "id" in row and not isinstance(row["id"], str):  # id is optional; validator flags id_type only when present
        probs.append("id_type")
    if row.get("role") not in roles_ok:
        probs.append("role_invalid")
    if "replay" in row and not isinstance(row["replay"], bool):
        probs.append("replay_flag")
    msgs = row.get("messages")
    if not isinstance(msgs, list) or len(msgs) != 3 or not all(isinstance(m, dict) for m in msgs):
        return probs + ["messages_shape"]
    if [m.get("role") for m in msgs] != ["system", "user", "assistant"]:
        probs.append("roles_order")
    if any(not isinstance(m.get("content"), str) or not m["content"].strip() for m in msgs):
        probs.append("content_type")
    if preamble_text is not None and str(msgs[0].get("content") or "").strip() != preamble_text:
        probs.append("preamble_mismatch")

    replay_tools = row.get("replay_tools")
    offered = None  # name -> set(parameter names)
    if replay_tools is not None:
        ok = isinstance(replay_tools, list) and replay_tools and all(
            isinstance(t, dict) and t.get("type") == "function" and isinstance(t.get("function"), dict)
            and isinstance(t["function"].get("name"), str) and isinstance(t["function"].get("parameters"), dict)
            for t in replay_tools)
        if ok:
            offered = {t["function"]["name"]: set((t["function"]["parameters"].get("properties") or {})) for t in replay_tools}
        else:
            probs.append("replay_tools_shape")

    content = str(msgs[2].get("content") or "")
    if content in fixed_replies:
        return probs
    if not (content.startswith("<tool_call>") and content.endswith("</tool_call>")):
        return probs + ["assistant_format"]
    calls = []
    for body in TAG.findall(content):
        try:
            call = json.loads(body)
        except json.JSONDecodeError:
            return probs + ["assistant_format"]
        if not isinstance(call, dict) or not isinstance(call.get("name"), str) or not isinstance(call.get("arguments"), dict):
            return probs + ["assistant_format"]
        calls.append(call)
    schema = offered if offered is not None else catalogue
    if schema is None:
        if replay_tools is None:
            probs.append("tools_required")
        return probs
    for call in calls:
        if call["name"] not in schema:
            probs.append("tool_unknown")
        elif offered is None and set(call["arguments"]) - schema[call["name"]]:
            probs.append("arg_unknown")  # argument keys are checked against the --tools catalogue only (as the validator does)
    return probs


OFFENDING = {}
for f in FILES:
    found, seen_ids = [], {}
    with (REPO / f).open(encoding="utf-8") as fh:
        for ln, line in enumerate(fh, 1):
            if not line.strip():
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                found.append((ln, "?", ["json_invalid"], line[:200]))
                continue
            probs = row_problems(row)
            rid = row.get("id") if isinstance(row, dict) else None
            if isinstance(rid, str):  # duplicates are tracked for string ids only, as the validator does
                if rid in seen_ids:
                    probs.append("id_duplicate")
                seen_ids.setdefault(rid, ln)
            if probs:
                found.append((ln, rid if rid is not None else "?", probs, line[:300]))
    OFFENDING[f] = found
    print(f"{f}: {len(found)} hàng nghi lỗi (pre-check)")
    for ln, rid, probs, raw in found[:3]:
        print(f"  line {ln}  id={rid}  -> {', '.join(probs)}")
        print(f"    {raw}...")

## Ý nghĩa từng loại lỗi và cách sửa

Tên loại lỗi đúng như `training/validate_dataset.py` in ra (script cũng in tối đa vài ví dụ `line N id=…` cho mỗi loại):

| Loại lỗi | Nghĩa là gì | Sửa thế nào |
|---|---|---|
| `json_invalid` | Dòng không phải JSON object hợp lệ (sửa tay, thiếu ngoặc, xuống dòng giữa hàng) | Sinh lại file bằng script; không sửa tay `.jsonl` |
| `id_type` / `id_duplicate` | `id` là **tuỳ chọn**; nếu có thì phải là chuỗi (`id_type`) và không trùng với hàng khác trong cùng file (`id_duplicate`, chỉ xét id dạng chuỗi) | Đặt id ổn định (`pub-…`, `sgod-…`) ngay ở bước sinh dữ liệu |
| `role_invalid` | Trường `role` không nằm trong tập cho phép (`employee`, `company_admin`, `system_admin` hoặc `roles` trong `tools/sgod/roles.json`) | Ánh xạ role theo `docs/contracts/roles.md`; không dùng ObjectId của SGOD |
| `replay_flag` | `replay` có mặt nhưng không phải `true`/`false` | Chỉ đặt `"replay": true` cho hàng có `replay_tools` |
| `messages_shape` | `messages` không phải list đúng 3 object `[system, user, assistant]` | Phase 2 chỉ đơn lượt: bỏ hàng đa lượt hoặc tách thành hàng đơn |
| `roles_order` | Thứ tự role không phải `system → user → assistant` | Kiểm tra bước tạo hàng (`build_parity_trainset.py` / `convert_xlam.py`) |
| `content_type` | Một message có `content` rỗng hoặc không phải chuỗi | Loại hàng có query rỗng ở bước lọc |
| `preamble_mismatch` | `messages[0].content` khác nội dung file preamble (so sau `.strip()`) — kể cả khác một dấu cách | Sinh lại với `--system-file` đúng phiên bản (v0 công khai, v1 SGOD); không tự sửa preamble trong hàng |
| `assistant_format` | Target không phải `<tool_call>{"name": str, "arguments": {…}}</tool_call>` (JSON hỏng, thiếu `name`, `arguments` không phải object, có khoá lạ) và cũng không đúng **nguyên văn** `REFUSAL_VI` / `DEFLECT_VI` | Chỉ có 3 dạng hợp lệ; tạo bằng `json.dumps(..., ensure_ascii=False)`; câu từ chối copy đúng từ `prompts/fixed_replies.json` |
| `replay_tools_shape` | `replay_tools` rỗng hoặc phần tử thiếu `type: "function"` / `function.name` / `function.description` / `function.parameters` (object) | Chuẩn hoá bằng `to_openai_tool()` trong `convert_xlam.py` |
| `tool_unknown` | Tên tool trong `<tool_call>` không có trong `replay_tools` (hoặc trong `--tools`) | Hàng bị lỗi nhãn → đưa vào quarantine, **không** sửa tên tool cho khớp |
| `tools_required` | Hàng không có `replay_tools` mà chạy không có `--tools` | Với hàng SGOD điền `TOOLS = "tools/sgod/sgod_tools.json"` ở ô Tham số |
| `arg_unknown` | (chỉ khi có `--tools`) `arguments` có khoá không nằm trong `parameters.properties` của tool trong danh mục | Sửa nhãn hoặc sửa schema tool (PR riêng, kèm `validate_tools.py`) |
| `tool_not_offered_to_role` | Tool được gọi nhưng `tool_policy.json` (cạnh `--tools`) không cho `role` của hàng dùng | Với role bị cấm, target phải là `REFUSAL_VI`, không phải `<tool_call>` |

Nguyên tắc: **sửa ở nguồn sinh dữ liệu rồi chạy lại**, không sửa tay từng dòng trong `.jsonl`.

## Tạo báo cáo

Copy khối in ra bên dưới (từ dòng `## Báo cáo …`) và dán vào **comment** của task ClickUp đang làm (Tuần 2 hoặc Việc 5).

In [ ]:
# Print the ClickUp comment block: per-file row counts, exit code, error summary from the validator.
import datetime

TASK = "Kiểm định bộ dữ liệu bằng validate_dataset.py (tiện ích cho Tuần 2 và Việc 5)"
lines = [
    f"## Báo cáo {TASK} — {datetime.date.today().isoformat()} — {AUTHOR}",
    f"- Notebook: `notebooks/data/03_validate_dataset.ipynb` @ `{GIT_SHA}` · môi trường: {'Colab' if IN_COLAB else 'local'}",
    "- Lệnh: `" + " ".join(["python", *cmd[1:]]) + "`",
    f"- Exit code `validate_dataset.py`: **{EXIT_CODE}** (0 = hợp lệ)",
]
for f in FILES:
    lines.append(f"- `{f}`: {ROW_COUNTS[f]} hàng · pre-check nghi lỗi: {len(OFFENDING[f])}")
if VALIDATOR_OUTPUT.strip():
    # Keep the per-file lines, the "errors by type" line and the final PASS/FAIL line of the validator.
    summary = [l.strip() for l in VALIDATOR_OUTPUT.splitlines()
               if l.startswith("[") or l.startswith("errors by type") or l.startswith("validate_dataset:")]
    lines.append("- Tóm tắt của validator:")
    lines += ["    - `" + l + "`" for l in summary]
print("\n".join(lines))